# Final Report - Fauxtography: Detecting AI-Generated Images

**Semester group project for MAI644 - Computer Vision, M.Sc. in Artificial
Intelligence, University of Cyprus, 2026.**

**Collaborators:** Manos (suprch4rg3d) and Demetra (DemetraP20).

This notebook is the joint deliverable: it reads the committed results under
`results/` and draws every figure inline. No model runs, no re-training -
each cell just opens a metrics JSON and plots it. If `results/` is present
(next to this notebook, or as a Drive mount), everything below works on a
laptop with plain `matplotlib`/`pandas`.


## Executive summary

Three detectors, one task: tell real photographs from AI-generated images on
the GenImage benchmark (Stable Diffusion v1.4 for training, plus three
never-seen generators for transfer). All results are committed and
re-runnable from this repo; every figure below reads them at render time.

| approach | same-generator (heldout) | best cross-generator |
|----------|--------------------------|----------------------|
| ResNet baseline (fine-tuned) | 0.9666 | Midjourney 0.5945 (then collapse to ~0.49) |
| DINOv2 + linear probe (frozen) | 0.8746 | Midjourney 0.8205 |
| CLIP zero-shot (best prompts) | 0.7277 | Midjourney 0.7885 (no training at all) |

Headlines: the paper's "training on one generator collapses on others" cliff
reproduces for our ResNet and does **not** fully replicate for DINOv2 - the
frozen foundation-model features keep Midjourney at 0.82 with zero
per-generator training. The detectors read **pixels, not bytes**: the 4x
file-size gap in GenImage is real, but removing it changes accuracy by less
than a point. Majority-vote fusion of all three families reaches 0.9392
heldout.

The honest framing (what we reproduced vs what we added) is in section 12;
the per-model verdicts are in the verdict section.


## The storyline

Three questions drive the whole project:

1. **Does it work on the distribution it trained on?** Can a fine-tuned
   ResNet, a zero-shot CLIP, and a frozen DINOv2 + linear probe separate
   Stable Diffusion v1.4 fakes from real photos, tested in-domain?
2. **Does the skill transfer?** Training sees only SD v1.4 fakes. What
   happens on fakes from generators the models never saw - Midjourney, ADM,
   BigGAN? The GenImage paper's headline is that a fine-tuned ResNet
   collapses here (~99.9% in-domain -> ~52-55% cross). Does that collapse
   replicate for foundation models?
3. **Is it the pixels or the bytes?** GenImage SD fakes are 4x larger files
   than real photos (median ~448 KB vs ~112 KB). A detector that just reads
   the size gap is not a fake detector. We probe this with a file-size
   threshold and a re-compression experiment that removes the size signal
   while keeping the pixels.


## Palette

Every figure in this notebook uses the same colors:

- **ResNet baseline** - slate `#94a3b8`
- **DINOv2 + probe** - violet `#7c3aed`
- **CLIP zero-shot** - cyan `#0891b2`
- **ensemble / fused** - pink `#ec4899`
- **recompressed (size removed)** - amber `#f59e0b`
- **chance line (0.50)** - red `#ef4444`


## 1. How the numbers were produced (the honest meta notes)

Where the experiments actually ran, what broke, and how we kept it
reproducible. This is the "runtime" section of the report.

**Hardware.** The committed run was executed on a **Colab L4 GPU** (the
`results/` on this repo came from the L4 run). The free-tier **T4** is
enough for all three models at project scale; only the baseline truly
trains, and the feature extraction is a single forward pass per image.
Device resolution is automatic: CUDA -> MPS (Apple Silicon) -> CPU, via
`utils/device.py`. Local smoke runs happened on an Apple M4 (Python 3.14,
torch 2.15).

**Environment from scratch, every session.** Colab ships Python 3.11/3.12;
this project pins Python 3.14 (`setup.md`), so each session installs `uv`,
then `uv venv --python 3.14 /content/fxt-venv`, then
`uv pip install --python /content/fxt-venv/bin/python -e .`.

**Nothing lives on the VM.** Colab sessions drop after ~90 minutes idle
(or ~12 h cap), so every input and output goes to Google Drive:
`FAUXTOGRAPHY_DATA_ROOT` / `RUN_DIR` / `RESULTS_DIR` env vars redirect the
config paths, and the final backup cell **force-copies** `results/` +
`experiments/` repo -> Drive with a tqdm progress bar, drops `.ready`
markers (so Drive's web UI stops hiding empty folders), then
`drive.flush_and_unmount()` + re-mount so writes reach Google's servers
immediately - the listing after that is read back from the cloud as ground
truth, not from the VM cache.

**Seeds and reproducibility.** Every config pins `seed: 42`; the split
is stratified and deterministic; the lockfile pins all versions. So the
numbers below are re-runnable - the whole point of a reference-point repo
(see `docs/contributions.md`).

**Cosmetic history.** Most of Demetra's commits are titled "Created using
Colab" - that is the Colab autosave convention, not content. The results
themselves are the controlled part.

**One process quirk.** Early runs used a plain square-resize preprocessing
for DINOv2; the canonical run uses official DINOv2 preprocessing (resize
short side to 256, center-crop 224). Both are committed
(`results/dinov2/` vs `results/dinov2_square/`): square gave 0.8887 /
0.8782, canonical 0.8873 / 0.8746. We kept canonical and report it.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Locate the repo root no matter where this notebook is opened from
# (repo root, notebooks/, or Colab's /content/fauxtography).
def _find_results(start: Path) -> Path:
    for d in (start, *start.parents):
        if (d / "results").is_dir():
            return d / "results"
    raise FileNotFoundError("results/ not found - open the notebook from the repo root")

results = _find_results(Path.cwd())

P_BASE = "#94a3b8"   # ResNet baseline (slate)
P_DINO = "#7c3aed"   # DINOv2 + probe (violet)
P_CLIP = "#0891b2"   # CLIP zero-shot (cyan)
P_FUSE = "#ec4899"   # ensemble (pink)
P_RECO = "#f59e0b"   # recompressed (amber)
P_C    = "#ef4444"   # chance (red)

def acc(*parts):
    """Read accuracy from a metrics.json under results/."""
    p = results.joinpath(*parts)
    if p.is_dir():
        p = p / "metrics.json"
    return json.loads(p.read_text())["accuracy"]

print("results root:", results)
print("setup OK")


## 2. The dataset

**GenImage** (Zhu et al., NeurIPS 2023 D&B, arXiv:2306.08571 - BibTeX in
`references.bib`): 2.68M images across eight generators. We use the
**Stable Diffusion v1.4** subset for the main train/val/test split, and
hold out **Midjourney**, **ADM** and **BigGAN** as cross-generator test
sets. Real side is the ImageNet "nature" subset; labels come from folder
names (`real/`, `fake/`) mapped to `real=0, fake=1`.

| split | images | real | fake |
|-------|--------|------|------|
| train | 14,000 | 7,000 | 7,000 |
| internal test | 3,000 | 1,500 | 1,500 |
| heldout (official test) | 12,000 | 6,000 | 6,000 |
| each cross generator | 2,000 | 1,000 | 1,000 |

Chance is 0.50 everywhere.

**The size gap that started a probe.** Dataset research noted, verbatim:
*"Very funny how for the same amount of training, 10001 each for real and
fake the real are 1.15GB and the fake are 4.58GB."* The real photos are
downscaled, heavily JPEG-compressed ImageNet images (median ~105-127 KB);
the fakes are generated natively at 512x512 and compress poorly (median
~448 KB). That 4x gap is the first thing a lazy detector can learn - and
it motivates the shortcut probes in section 8.


### What we are classifying

Six real photographs and six SD v1.4 fakes from the internal test split.
The real side is ImageNet nature photos, downscaled and heavily
JPEG-compressed; the fakes are generated natively at 512x512. The class
distinction is often in texture, compression and geometry, not in "what
the subject is".


In [ ]:
# Sample grid: real vs SD v1.4 fake
def _find_dir(start: Path, name: str) -> Path | None:
    for d in (start, *start.parents):
        if (d / name).is_dir():
            return d / name
    return None

from PIL import Image

data_root = _find_dir(Path.cwd(), "data")

def _grid(paths, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.4))
    for ax, p in zip(axes, paths):
        im = np.asarray(Image.open(p).convert("RGB").resize((224, 224)))
        ax.imshow(im); ax.axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

if data_root is not None and (data_root / "test" / "real").is_dir():
    real = sorted((data_root / "test" / "real").glob("*.jpg"))[:6]
    fake = sorted((data_root / "test" / "fake").glob("*.jpg"))[:6]
    _grid(real, "REAL photographs (ImageNet nature subset)")
    _grid(fake, "FAKE - Stable Diffusion v1.4")
else:
    print("data/test not found next to this notebook - sample grid skipped "
          "(run where the dataset is mounted, e.g. Colab with Drive)")


### 2.1 Preprocessing A/B for DINOv2 (a robustness check)

Early runs used a plain square resize; the canonical run uses the official
DINOv2 preprocessing (resize short side to 256, center-crop 224). Both are
committed - the point of an A/B is to show the choice is not load-bearing:


In [ ]:
sq = pd.DataFrame([
    {"variant": "canonical", "internal": acc("dinov2", "metrics.json"),
     "heldout": acc("dinov2", "heldout_metrics.json")},
    {"variant": "square", "internal": acc("dinov2_square", "metrics.json"),
     "heldout": acc("dinov2_square", "heldout_metrics.json")},
]).set_index("variant")
sq.round(4)


In [ ]:
# Same numbers, as a figure
x = np.arange(len(sq)); w = 0.28
fig, ax = plt.subplots(figsize=(6, 4), dpi=200)
b1 = ax.bar(x - w/2, sq["internal"], w, label="internal", color=P_DINO)
b2 = ax.bar(x + w/2, sq["heldout"], w, label="heldout", color=P_FUSE)
for bars in (b1, b2):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.005,
                f"{h:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(sq.index)
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.0)
ax.legend(frameon=False)
ax.set_title("DINOv2 preprocessing: canonical vs square", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## 3. The three approaches

| approach | model | training we do |
|----------|-------|----------------|
| baseline | ResNet-18 (fine-tuned CNN) | full fine-tune, 12 epochs, best-val restore (`train-baseline`) |
| CLIP | `openai/clip-vit-base-patch32` | none - zero-shot text prompts (`eval-clip`) |
| DINOv2 | `dinov2_vitb14` frozen + one linear layer | linear probe on cached [CLS] features, 20 epochs (`extract-dinov2` + `train-linear-probe`) |

Fairness caveat, stated once and repeated in the verdict: CLIP is a
**zero-shot reference point**, not an equal-footing competitor. It never
sees a GenImage training sample, while ResNet is fine-tuned on them and
DINOv2's probe is trained on them. The interesting question is not "which
wins" but "what each regime buys".

All three write the same `results/<model>/metrics.json` layout, so
`compare` and `analyze-fusion` treat them identically.


## 4. In-domain results: does each approach work at all?

Read straight from the committed metrics. Internal test = the 3,000-image
SD v1.4 split; heldout = the 12,000-image official SD v1.4 test.


In [ ]:
df = pd.DataFrame([
    {"model": "ResNet baseline", "internal": acc("baseline", "metrics.json"),
     "heldout": acc("baseline", "heldout_metrics.json")},
    {"model": "DINOv2 + probe", "internal": acc("dinov2", "metrics.json"),
     "heldout": acc("dinov2", "heldout_metrics.json")},
    {"model": "CLIP zero-shot", "internal": acc("clip", "metrics.json"),
     "heldout": acc("clip", "heldout_metrics.json")},
]).set_index("model")
df.round(4)


In [ ]:
# Figure 1 - In-domain accuracy: the three approaches, internal vs heldout
x = np.arange(len(df))
w = 0.32
fig, ax = plt.subplots(figsize=(9, 4.6), dpi=200)
b1 = ax.bar(x - w/2, df["internal"], w, label="internal test (n=3000)", color=P_BASE)
b2 = ax.bar(x + w/2, df["heldout"], w, label="official heldout (n=12000)", color=P_DINO)
ax.axhline(0.5, color=P_C, ls="--", lw=1.2)
ax.text(2.4, 0.51, "0.50 = chance", color=P_C, fontsize=9)
for bars in (b1, b2):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.006,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(df.index, fontsize=10)
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.06)
ax.legend(loc="lower left", frameon=False)
ax.set_title("In-domain SD v1.4 detection", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


**Reading.** ResNet (0.970/0.967) is the strongest in-domain, exactly as
expected for a fully fine-tuned CNN. DINOv2 + one linear layer (0.887 /
0.875) is close behind with ~0 trainable parameters of backbone and no
image-level fine-tuning - the features do the work. CLIP zero-shot
(0.713 / 0.728, its best prompt strategy) is well above chance but far
below either trained model: prompt wording is doing the heavy lifting, not
any GenImage-specific knowledge.

**Stability check.** Internal vs heldout gaps are small for every model
(ResNet 0.004, DINOv2 0.013, CLIP 0.015), so nothing is overfit to the
3,000-image internal split. The numbers transfer to the fresh 12,000.


### 4.1 Per-class detail (precision / recall / F1)

Accuracy hides the error direction. Same committed metrics, per class:


In [ ]:
def _prf(model):
    d = json.loads((results / model / "metrics.json").read_text())
    r = d["classification_report"]
    out = {}
    for cls_ in ("real", "fake"):
        for m in ("precision", "recall", "f1-score"):
            out[f"{cls_} {m}"] = r[cls_][m]
    return out

pdf = pd.DataFrame({m: _prf(m) for m in ("baseline", "dinov2", "clip")}).T
pdf.round(4)


## 5. Confusion matrices (heldout)

Where each model errs, on the same 12,000 heldout images (6,000 real +
6,000 fake). Committed SVGs are displayed directly.


In [ ]:
from IPython.display import SVG, display

for name, title, color in [
    ("baseline", "ResNet baseline", P_BASE),
    ("dinov2", "DINOv2 + probe", P_DINO),
    ("clip", "CLIP zero-shot", P_CLIP),
]:
    p = results / name / "heldout_confusion_matrix.svg"
    if p.exists():
        print(f"--- {title} ---")
        display(SVG(filename=str(p)))
    else:
        print(f"no heldout confusion matrix for {name}")


**Reading.** One pattern runs through all three: every model misses fakes
more than it over-flags reals. ResNet errs least (148 real-as-fake vs 253
fake-as-real on heldout - ~2% total), DINOv2 errs more and with the same
lean (883 vs 622), and CLIP at ~0.73 is the only one with visually
substantial confusion - yet it is also the most symmetric (1598 vs 1670,
ratio 1.05), which matches its symmetric zero-shot priors. So the class
bias (fakes slip through more) is strongest exactly where accuracy is
highest, and washes out as the model gets weaker.


## 6. Error inspection: which images confuse which model

The manual inspection the docs call the core of the presentation. For each
model we take the committed predictions, find the mistakes on the internal
test split, and show the actual images: real photographs flagged as fake,
and fake images that slipped through as real.

**Label mapping, stated precisely.** PyTorch's `ImageFolder` maps class
folders alphabetically (`fake` < `real`, so fake=0 / real=1 natively), but
our `FauxtoDataset` wrapper remaps to the project's canonical order
(real=0 / fake=1) when it returns labels. The eval commands that wrote
`y_true/y_pred` use that wrapper, so the arrays carry real=0 / fake=1.
Index i in the arrays corresponds to `dataset.folder.samples[i]` because
eval and this cell construct the same `FauxtoDataset` in the same order.
The code cell verifies this alignment at runtime (and would fail loudly
if a future data change ever broke it). If the split is not mounted next
to this notebook, the cell prints a message instead of failing.


In [ ]:
from PIL import Image
from fauxtography.data.dataset import FauxtoDataset

data_root = _find_dir(Path.cwd(), "data")

def _row(model, title, mask):
    ds = FauxtoDataset(data_root / "test")
    yt = np.load(results / model / "y_true.npy")
    yp = np.load(results / model / "y_pred.npy")
    # Guard against the ImageFolder alphabetical-mapping gotcha: verify that
    # y_true (real=0/fake=1, written through FauxtoDataset when eval ran) is
    # actually index-aligned with this dataset's samples. If a future change
    # ever desyncs them, fail loudly instead of showing wrong images.
    if len(yt) != len(ds):
        raise RuntimeError(
            f"{model}: y_true has {len(yt)} rows but dataset has {len(ds)} "
            "samples - alignment broken, check the data split"
        )
    dense_labels = np.array([ds[i][1] for i in range(len(ds))])
    if not (dense_labels == yt).all():
        raise RuntimeError(
            f"{model}: y_true labels do not match the dataset's class order - "
            "FauxtoDataset mapping (real=0/fake=1) vs ImageFolder default "
            "changed. Fix before trusting these figures."
        )
    idxs = [i for i, (t, p) in enumerate(zip(yt, yp)) if mask(t, p)][:4]
    if not idxs:
        print(f"{model}: no examples for '{title}'"); return
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.3 * len(idxs), 2.6))
    if len(idxs) == 1:
        axes = [axes]
    for ax, i in zip(axes, idxs):
        path = ds.folder.samples[i][0]
        im = np.asarray(Image.open(path).convert("RGB").resize((224, 224)))
        ax.imshow(im); ax.axis("off")
    fig.suptitle(f"{model}: {title}", fontsize=11)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

if data_root is not None and (data_root / "test").is_dir():
    for m in ("baseline", "dinov2", "clip"):
        _row(m, "real photo flagged as FAKE", lambda t, p: t == 0 and p == 1)
        _row(m, "fake image passed as REAL", lambda t, p: t == 1 and p == 0)
else:
    print("data/test not found next to this notebook - error inspection skipped")


## 7. Cross-generator generalization: the generator cliff

The GenImage paper's core result, reproduced with our ResNet and extended
with DINOv2 and CLIP. The probe / fine-tuned model sees **only SD v1.4**
fakes; the cross rows test it on never-seen generators (2,000 images each,
1,000 real + 1,000 fake). Paper row from arXiv:2306.08571 (ResNet-50
trained on the full SD v1.4 pool - see the honesty caveat).

| Tested on | Paper ResNet-50 | Our ResNet | DINOv2 | CLIP |
|-----------|-----------------|------------|--------|------|
| SD v1.4 (heldout) | ~99.9% | 0.9666 | 0.8746 | 0.7277 |
| Midjourney | ~54.9% | 0.5945 | 0.8205 | 0.7885 |
| ADM | ~53.5% | 0.4910 | 0.5100 | 0.7035 |
| BigGAN | ~52.0% | 0.4950 | 0.5545 | 0.7565 |

Every cross row is 2,000 images; 0.50 = chance. The CLIP cross cells
come from a real zero-shot run (results/clip/cross/<generator>/) - and
they are the surprise of this table: a model with zero GenImage training
holds 0.70-0.79 on generators it never saw, above the paper's fine-tuned
ResNet-50 row and close to DINOv2.


The full paper matrix (ResNet-50, trained row / tested column, % accuracy) -
the numbers we verified against arXiv:2306.08571 and reproduce in shape with
our smaller ResNet:

| Trained / Tested | SD v1.4 | Midjourney | ADM | BigGAN | VQDM |
|-------------------|---------|------------|-----|--------|------|
| **SD v1.4** | 99.9 | 54.9 | 53.5 | 52.0 | 56.6 |
| **Midjourney** | 76.4 | 98.8 | 64.1 | 50.1 | 52.5 |
| **ADM** | 53.1 | 58.6 | 99.0 | 88.3 | 61.5 |
| **BigGAN** | 49.9 | 49.9 | 50.6 | 99.9 | 50.6 |

Every model family memorizes its own generator (the diagonal) and slumps
elsewhere. Our contribution is to re-run the SD-trained row against two
model families that are not fine-tuned CNNs - figure 2 below.


In [ ]:
# Figure 2 - The generator cliff: DINOv2 vs ResNet across populations
groups = ["train\n(internal)", "heldout\n(SD v1.4)", "Midjourney", "ADM", "BigGAN"]
din = [
    acc("dinov2", "metrics.json"),
    acc("dinov2", "heldout_metrics.json"),
    acc("dinov2", "cross", "Midjourney", "metrics.json"),
    acc("dinov2", "cross", "ADM", "metrics.json"),
    acc("dinov2", "cross", "BigGAN", "metrics.json"),
]
base = [
    acc("baseline", "metrics.json"),
    acc("baseline", "heldout_metrics.json"),
    acc("baseline", "cross", "Midjourney", "metrics.json"),
    acc("baseline", "cross", "ADM", "metrics.json"),
    acc("baseline", "cross", "BigGAN", "metrics.json"),
]
clip_cross = [
    acc("clip", "metrics.json"),
    acc("clip", "heldout_metrics.json"),
    acc("clip", "cross", "Midjourney", "metrics.json"),
    acc("clip", "cross", "ADM", "metrics.json"),
    acc("clip", "cross", "BigGAN", "metrics.json"),
]

x = np.arange(len(groups))
w = 0.27
fig, ax = plt.subplots(figsize=(11, 5.4), dpi=200)
b1 = ax.bar(x - w, base, w, label="ResNet baseline", color=P_BASE)
b2 = ax.bar(x, din, w, label="DINOv2 + probe", color=P_DINO)
b3 = ax.bar(x + w, clip_cross, w, label="CLIP zero-shot", color=P_CLIP)
ax.axhline(0.5, color=P_C, ls="--", lw=1.2)
ax.text(4.42, 0.52, "0.50 = chance", color=P_C, fontsize=9)
for bars in (b1, b2, b3):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.008,
                f"{h:.3f}", ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=9)
ax.axvline(1.5, color="black", lw=0.6, alpha=0.4)
ax.text(0.72, 0.98, "SEEN\n(SD v1.4)", ha="center", va="top", fontsize=8, color="#555")
ax.text(3.3, 0.98, "UNSEEN GENERATORS", ha="center", va="top", fontsize=8, color="#555")
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.07)
ax.legend(loc="upper left", frameon=False)
ax.set_title("Cross-generator detection: DINOv2 probe vs ResNet baseline "
             "(no per-generator training)", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


In [ ]:
# Cross-generator confusion matrices: how the errors distribute per generator
from IPython.display import SVG as _SVG4, display as _display4
gens = ["Midjourney", "ADM", "BigGAN"]
for model, color in (("baseline", P_BASE), ("dinov2", P_DINO), ("clip", P_CLIP)):
    for g in gens:
        p = results / model / "cross" / g / "confusion_matrix.svg"
        if p.exists():
            print(f"{model} on {g}:")
            _display4(_SVG4(filename=str(p)))
        else:
            print(f"missing {p}")


**Reading.** The fine-tuned ResNet **reproduces the paper's cliff**: 0.97
in-domain, then 0.59 on Midjourney and ~0.49 on ADM/BigGAN - its
same-generator edges are memorized, not transferable. DINOv2 + probe
**holds much better on Midjourney (0.82 vs 0.59)** and edges the baseline
on ADM (0.51 vs 0.49) and BigGAN (0.55 vs 0.50), with no retraining.
Frozen self-supervised features absorb "natural image structure" that
transfers; a fully fine-tuned CNN latches onto the generator's look.

The paper's ~54.9/53.5/52.0 row (full SD pool, ResNet-50) is reproduced
in *shape* by our smaller ResNet (0.59/0.49/0.495). Our numbers are not
directly comparable in level - we trained on a ~7k-fake slice vs the
paper's full pool - but the collapse pattern is the same, now shown for a
second model family.

**CLIP is the quiet surprise.** Zero-shot, no GenImage training at all,
and it still holds 0.7885 / 0.7035 / 0.7565 across the unseen generators -
above the paper's fine-tuned ResNet-50 row everywhere and, on ADM and
BigGAN, above even DINOv2 (0.70 vs 0.51, and 0.76 vs 0.55). A model that
never memorized any generator's look has no same-generator pedestal to
lose, so it generalizes by default. Its in-domain 0.7277 is the lowest of
the three, but its *relative* cross-gen stability is the best of the
table.


## 8. CLIP: prompt wording is a hyperparameter

CLIP gets no training, so the only knobs are the **text prompts**. Demetra
swept the full strategy family (all results committed under
`results/clip/<strategy>/`, same images per split). One table says it all:


In [ ]:
import glob as _glob
rows = []
for d in sorted(_glob.glob(str(results / "clip" / "*"))):
    dpath = Path(d)
    if not (dpath / "metrics.json").exists():
        continue
    m = json.loads((dpath / "metrics.json").read_text())
    h = json.loads((dpath / "heldout_metrics.json").read_text())
    rows.append({
        "strategy": dpath.name,
        "internal": m["accuracy"],
        "heldout": h["accuracy"],
        "internal macro-F1": m["classification_report"]["macro avg"]["f1-score"],
        "heldout macro-F1": h["classification_report"]["macro avg"]["f1-score"],
    })
sweep = (pd.DataFrame(rows)
           .sort_values("heldout", ascending=False)
           .reset_index(drop=True))
sweep


In [ ]:
# Figure 3 - The prompt sweep: heldout accuracy by strategy
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=200)
y = np.arange(len(sweep))
cols = [P_CLIP if s == "balanced-descriptions" else P_BASE for s in sweep["strategy"]]
ax.barh(y, sweep["heldout"], color=cols, height=0.62)
for yi, h in zip(y, sweep["heldout"]):
    ax.text(h + 0.004, yi, f"{h:.3f}", va="center", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(sweep["strategy"], fontsize=9)
ax.invert_yaxis()
ax.axvline(0.5, color=P_C, ls="--", lw=1)
ax.set_xlabel("heldout accuracy"); ax.set_xlim(0, 0.85)
ax.set_title("CLIP prompt sweep (all zero-shot, same images)",
             fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


**Reading.** The same frozen CLIP moves from **0.48 to 0.73 heldout just
by changing prompt wording** - the backbone is constant, only the text
changes. The active config is **"balanced descriptions"** (internal 0.7130
/ heldout 0.7277), the sweep winner. The pre-sweep "original baseline"
prompts scored 0.665 / 0.6647, and the "combined" ensemble mentioned in
early docs scored 0.6447 / 0.6581. This is a real finding: for zero-shot
CLIP, prompt design matters at least as much as model choice, and any
single CLIP number is meaningless without its prompt set. It also means
CLIP's 0.73 is a *tuned* zero-shot number - we state that, not hide it.


## 9. Is it the pixels or the bytes?

Two probes on the same question. The corpus gap: real median ~112-127 KB,
fake ~448 KB (4x). If a model were reading that, it would not be a fake
detector.

**8a. File-size threshold probe.** A trivial rule ("file bigger than a
threshold => fake") scores surprisingly high on the sets whose fakes are
large - and falls to ~0.49 on generators whose fakes compress well:


In [ ]:
fs = json.loads((results / "file_size" / "metrics.json").read_text())
print(f"rule: fake > {fs['threshold_kb']:.1f} KB => fake")
print(f"mean sizes: real {fs['mean_size_bytes']['real_train_kb']:.1f} KB, "
      f"fake {fs['mean_size_bytes']['fake_train_kb']:.1f} KB")
pd.DataFrame([
    {"set": s["name"], "n": s["n_samples"],
     "file-size accuracy": round(s["accuracy"], 4)}
    for s in fs["sets"]
])


In [ ]:
# The committed histogram of the size gap (real vs fake, train split)
from IPython.display import SVG as _SVG2, display as _display2
hist = results / "file_size" / "size_histogram.svg"
if hist.exists():
    print("File-size histogram (from the train split, committed by probe-file-size):")
    _display2(_SVG2(filename=str(hist)))
else:
    print("no size_histogram.svg on disk")


So a pure size rule reaches **0.99 on SD train/test and Midjourney** but
**~0.49 on ADM/BigGAN** (where it calls every fake "real", 0 true
positives - see the committed JSON). The size signal is real, it is not
the whole story, and its presence makes clean in-domain numbers
suspicious until checked.

**8b. The causal probe: re-compress and re-evaluate.** Re-encode the
heldout test so both classes land at a similar file size (real median
127 -> 87 KB, fake median 458 -> 95 KB - the gap is destroyed), re-extract
frozen DINOv2 features, and score the *unchanged* probe. If accuracy
collapses, the detector was using the size gap; if it holds, the model
reads pixels.


In [ ]:
# Figure 4 - Compression probe: destroy the size signal, keep the pixels
orig = acc("dinov2", "heldout_metrics.json")
c = json.loads((results / "dinov2" / "compression" / "metrics.json").read_text())
recom = c["accuracy"]
before, after = c["median_bytes_before"], c["median_bytes_after"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 4.2), dpi=200,
                               gridspec_kw={"width_ratios": [1, 1.25]})

ax1.bar(["original\nheldout", "recompressed\nto real size"],
        [orig, recom], color=[P_DINO, P_RECO], width=0.55)
for i, h in enumerate((orig, recom)):
    ax1.text(i, h + 0.006, f"{h:.3f}", ha="center", va="bottom", fontsize=10)
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("accuracy")
ax1.axhline(0.5, color=P_C, ls="--", lw=1)
ax1.set_title("Accuracy survives re-compression", fontsize=10)
ax1.set_xticks(range(2))
ax1.set_xticklabels(["original", "recompressed"], fontsize=8)
ax1.spines[["top", "right"]].set_visible(False)

ax2.bar(["real", "fake"],
        [before["real"]/1024, before["fake"]/1024],
        width=0.4, color=P_BASE, label="original")
ax2.bar([i + 0.42 for i in range(2)],
        [after["real"]/1024, after["fake"]/1024],
        width=0.4, color=P_RECO, label="recompressed")
ax2.set_xticks(range(2)); ax2.set_xticklabels(["real", "fake"])
ax2.set_ylabel("median file size (KB)")
for i, (b, a) in enumerate(zip(
        [before["real"]/1024, before["fake"]/1024],
        [after["real"]/1024, after["fake"]/1024])):
    ax2.text(i, b + 6, f"{b:.0f}", ha="center", fontsize=9)
    ax2.text(i + 0.42, a + 6, f"{a:.0f}", ha="center", fontsize=9, color=P_RECO)
ax2.set_ylim(0, 540)
ax2.legend(frameon=False)
ax2.set_title("Size gap destroyed", fontsize=10)
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle("Compression probe: not a file-size cheat", fontsize=12)
plt.show()


In [ ]:
# The same probe, as a confusion matrix: where does the recompressed probe err?
from IPython.display import SVG as _SVG3, display as _display3
cm_p = results / "dinov2" / "compression" / "confusion_matrix.svg"
if cm_p.exists():
    print("Recompressed probe confusion matrix (heldout):")
    _display3(_SVG3(filename=str(cm_p)))
else:
    print("no compression confusion matrix on disk")


**Reading.** Accuracy moves 0.8746 -> 0.8678 with the size gap removed -
a 0.7-point dip, essentially noise-level for a probe with n=12,000. The
DINOv2 detector reads **pixels, not bytes**. This is the causal claim the
file-size correlation cannot make, and it is Contribution 5 in
`docs/contributions.md`. (The broken 15,701-sample recompression attempt
was excluded and redone at the correct 12,000; the committed number is
the good one.)


## 10. What the feature space looks like

The frozen DINOv2 features, projected with PCA -> t-SNE and colored by
generator (8 classes, 8,000 points, committed via `visualize-features`):


In [ ]:
from IPython.display import SVG as _SVG, display as _display
emb = results / "dinov2" / "embedding" / "embedding.svg"
if emb.exists():
    meta = json.loads((results / "dinov2" / "embedding" / "embedding_classes.json").read_text())
    _display(_SVG(filename=str(emb)))
    print(f"DINOv2 feature space (PCA->t-SNE, {meta['n_points']} points, "
          f"{len(meta['classes'])} classes)")
else:
    print("no embedding on disk yet")


**Reading.** Real images from train and the three cross-generator sets
cluster together - the model's notion of "real" is stable across data
sources. SD v1.4 fakes form their own cluster (the detector's in-domain
signal), and each generator's fakes sit apart from it - Midjourney's
visibly nearest (it is the generator DINOv2 scores best on, 0.82), ADM
and BigGAN further out. That geometry is the cliff in Figure 2, visible
before any probe is trained: cross-generator collapse is in the
representation itself, not just in the fine-tuned head.


## 11. Fusion: does combining the families help?

`analyze-fusion` majority-votes the three model families (any two agree
=> that label wins) on the same splits and reports pairwise agreement.


In [ ]:
frows = json.loads((results / "fusion" / "summary.json").read_text())
fdf = pd.DataFrame([
    {"split": r["split"],
     "baseline": r["model_accuracies"]["baseline"],
     "dinov2": r["model_accuracies"]["dinov2"],
     "clip": r["model_accuracies"]["clip"],
     "fused (vote)": r["fused_accuracy"],
     "agree baseline+dinov2": r["pairwise_agreement"]["baseline_vs_dinov2"],
     "agree baseline+clip": r["pairwise_agreement"]["baseline_vs_clip"],
     "agree dinov2+clip": r["pairwise_agreement"]["dinov2_vs_clip"]}
    for r in frows
])
fdf.round(4)


In [ ]:
# Figure 5 - The ensemble: all three families, majority vote
x = np.arange(len(fdf))
w = 0.2
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=200)
ax.bar(x - 1.5*w, fdf["baseline"], w, label="ResNet baseline", color=P_BASE)
ax.bar(x - 0.5*w, fdf["dinov2"], w, label="DINOv2 + probe", color=P_DINO)
ax.bar(x + 0.5*w, fdf["clip"], w, label="CLIP zero-shot", color=P_CLIP)
ax.bar(x + 1.5*w, fdf["fused (vote)"], w, label="fused (majority vote)", color=P_FUSE)
for xi, row in fdf.iterrows():
    for j, key in enumerate(["baseline", "dinov2", "clip", "fused (vote)"]):
        h = row[key]
        ax.text(xi + (j - 1.5)*w, h + 0.004, f"{h:.3f}",
                ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(fdf["split"])
ax.axhline(0.5, color=P_C, ls="--", lw=1)
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.14)
ax.legend(frameon=False, loc="upper right", fontsize=9)
ax.set_title("Majority-vote fusion of ResNet + DINOv2 + CLIP", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


**Reading.** Fused accuracy is 0.9483 internal / **0.9392 heldout** - above
DINOv2 and CLIP alone, below the baseline alone. The cheap lesson: a
weak third voter mostly adds ties it then decides, so the ensemble gains
robustness but not a higher ceiling. The more useful signal is pairwise
agreement: baseline and DINOv2 agree ~0.86-0.87 (they are the two trained
models), while CLIP parts company 0.68-0.72 (its pairwise agreement with
either trained model). So the fusion figure says
less than the disagreement map - where two very different training
regimes still agree, the label is trustworthy.


## 12. What we reproduced, what we added, what we did not claim

The honesty rule from `docs/contributions.md` applies here verbatim. The
cross-generator experiment is **GenImage's** experiment; we never claim it
as new. What *we* added:

1. **The model axis.** The paper only evaluated fine-tuned CNNs. We ran the
   identical protocol on DINOv2 (self-supervised features + probe) and CLIP
   (zero-shot). The finding: the paper's collapse is not CNN-only -
   DINOv2 drops to 0.51/0.55 on ADM/BigGAN too - but it is *model-
   dependent*: DINOv2 holds Midjourney (0.82 vs 0.59 baseline), and CLIP
   - with zero training - holds 0.70-0.79 on every unseen generator. The
   memorizing CNN (0.59/0.49/0.495) is the one that cannot keep its
   accuracy away from home.
2. **The interpretation axis.** Manual error inspection turns the numbers
   into a story: which real images fool each model, which fake styles are
   most confusable, how the confusion cells differ per generator.
3. **The shortcut axis (causal, not correlational).** The 4x file-size gap
   is a documented GenImage flaw; our re-compression probe shows the
   detectors read pixels (0.875 -> 0.868), which the size-probe alone
   cannot claim.
4. **Feature geometry.** The t-SNE shows the collapse exists in the
   representation, before any head is trained.
5. **The comparison table as a reproducible artifact.** Fixed seeds,
   committed results, cheap re-runs: the table is a reference point others
   can extend (see README, "Adding another model").

What we did not claim: reproducing the paper, same-generator accuracy at
smaller scale (our ~7k-fake slice vs the paper's full pool), the exact
accuracies as standalone findings.

We verified the paper's published numbers against the actual paper
(arXiv:2306.08571, HTML full text) before citing: the table (99.9 /
54.9 / 53.5 / 52.0 for SD-trained ResNet-50) and the full matrix match
`docs/dataset_research.md` cell-for-cell. The BibTeX in `references.bib`
is validated with `bibtex` (zero errors).


## 13. Verdict

Three models, one question, three different answers - and together they
tell the story:

- **ResNet baseline (0.9666 heldout).** The reference point the paper
  already gave us, reproduced at smaller scale. In-domain king,
  cross-generator collapse (0.59 / 0.49 / 0.495). A fine-tuned CNN
  memorizes the generator's look. Its value here is as the *known* anchor
  and the strongest single number in-domain.

- **DINOv2 + linear probe (0.8746 heldout, 0.8205 Midjourney).** The most
  informative model. No image-level fine-tuning, yet near-ResNet in-domain
  and a real generalization edge on Midjourney - self-supervised features
  transfer, memorized edges do not. It still trips on ADM/BigGAN (0.51 /
  0.55) - the problem is harder than features alone. It is explicitly not
  a file-size shortcut (0.868 when recompressed).

- **CLIP zero-shot (0.7277 heldout, tuned).** The hypothesis-driven
  wildcard: zero GenImage training, zero backbone tuning, and still
  clearly above chance with the right prompts. Its sweep (0.48 to 0.73
  by wording alone) is a finding in itself - for zero-shot vision-language
  models, the prompt IS the classifier.

**Joint takeaway.** For deployment, the fine-tuned ResNet wins in-domain
and the DINOv2 probe is the generalization buy; the majority vote of all
three (0.9392 heldout) is the robust artifact, and the disagreement map
shows where an ensemble earns its keep. For the science: the GenImage
collapse is not an artifact of CNNs or fine-tuning - it is intrinsic to
cross-generator fake detection - but model choice shifts *which* unseen
generators bite, and the file-size flaw in GenImage does not explain the
collapse away.

**One sentence for the slides:** *A memorizing CNN wins at home and
collapses away; frozen self-supervised features travel better; zero-shot
CLIP is prompt-sensitive and above chance; and none of them are reading
the bytes.*

**Caveats.** Cross-generator eval uses one SD-trained probe with no
per-generator training (the paper trains a ResNet per generator). Our
training slice (~7k fakes) is much smaller than the paper's pool, so our
absolute numbers are not directly comparable to the paper's - the
*pattern* is. The file-size probe shows
a real shortcut signal exists in GenImage; we showed our detectors do not
depend on it, not that GenImage is free of it.


## 14. Future work

What a follow-up could (and should) do:

- **CLIP cross-generator cells are done** (0.7885 / 0.7035 / 0.7565, zero-
  shot). Natural next: re-run the prompt-sweep winner on the cross sets to
  see whether prompt choice moves the cross rows as much as it does the
  in-domain one.
- **More foundations.** The repo is built for it (fixed seeds, committed
  results, `compare`/`analyze-fusion` read any new `results/<model>/`
  layout): swap in SigLIP, a larger CLIP, or another self-supervised ViT
  and the table extends itself.
- **Compression-aware training.** Normalize file size / JPEG quality at
  data-prep time (see `docs/dataset_research.md`) so in-domain numbers
  are not inflated by the size shortcut at all.
- **Calibration, not just accuracy.** The disagreement map (section 11)
  shows where models part company; a confidence-calibrated ensemble could
  use those ties instead of voting through them.
- **Real-world stress.** Newer generators (SDXL, Flux, video frames) and
  images that went through screenshots / re-uploads are the deployment
  test this benchmark does not cover.


## 15. References and how to re-run everything

Papers:

- **GenImage: A Million-Scale Benchmark for Detecting AI-Generated
  Image.** Zhu, Chen, Yan, Huang, Lin, Li, Tu, Hu, Hu, Wang. NeurIPS 2023
  (Datasets and Benchmarks Track), arXiv:2306.08571. BibTeX in
  `references.bib`.
- **DINOv2: Learning Robust Visual Features without Supervision.** Oquab
  et al., 2023.
- **Learning Transferable Visual Models From Natural Language
  Supervision (CLIP).** Radford et al., 2021.
- Related bias analysis: "Fake or JPEG? Revealing Common Biases in
  Generated Image Detection Datasets" (UnbiasedGenImage) - the compressed
  shortcut context.

Re-running the whole thing takes three commands on a Colab GPU (one
notebook: `notebooks/fauxtography-on-colab.ipynb`), then the analysis
notebooks read the results:

```
fauxtography train-baseline
fauxtography eval-clip
fauxtography extract-dinov2 && fauxtography train-linear-probe
fauxtography compare && fauxtography analyze-fusion
```

All numbers above are read from committed JSONs at render time - rerun
this notebook after any future model joins and the figures update by
themselves (that is the reference-point design).


## 16. Appendix: the complete metric inventory

Nothing is left on the cutting-room floor; every committed metrics JSON is
read into a table below. Each row keeps accuracy, the per-class
precision / recall / F1 / support, and the macro and weighted aggregates -
the fields the comparison elsewhere trims to accuracy or headline F1.


In [ ]:
import glob as _g2

def _full_report(d):
    r = d.get("classification_report", {})
    def row_for(cls_):
        c = r.get(cls_, {})
        return {"precision": c.get("precision"), "recall": c.get("recall"),
                "f1": c.get("f1-score"), "support": c.get("support")}
    return {
        "accuracy": r.get("accuracy", d.get("accuracy")),
        "real": row_for("real"),
        "fake": row_for("fake"),
        "macro_f1": r.get("macro avg", {}).get("f1-score"),
        "weighted_f1": r.get("weighted avg", {}).get("f1-score"),
    }

def _main_rows():
    rows = []
    for model in ("baseline", "dinov2", "clip"):
        for label, fname in (("internal", "metrics.json"),
                             ("heldout", "heldout_metrics.json")):
            p = results / model / fname
            if not p.exists():
                continue
            d = json.loads(p.read_text())
            rep = _full_report(d)
            rows.append({
                "model": model, "split": label, "n": d.get("n_samples", ""),
                "accuracy": rep["accuracy"],
                "real P": rep["real"]["precision"], "real R": rep["real"]["recall"],
                "real F1": rep["real"]["f1"],
                "fake P": rep["fake"]["precision"], "fake R": rep["fake"]["recall"],
                "fake F1": rep["fake"]["f1"],
                "macro F1": rep["macro_f1"], "weighted F1": rep["weighted_f1"],
            })
    return pd.DataFrame(rows)

_main_rows().round(4)


In [ ]:
# Cross-generator detail (baseline + DINOv2, all three generators)
cross_rows = []
for model in ("baseline", "dinov2", "clip"):
    for g in ("Midjourney", "ADM", "BigGAN"):
        d = json.loads((results / model / "cross" / g / "metrics.json").read_text())
        r = d.get("classification_report", {})
        cross_rows.append({
            "model": model, "generator": g, "n": d.get("n_samples"),
            "accuracy": d["accuracy"],
            "real F1": r.get("real", {}).get("f1-score"),
            "fake F1": r.get("fake", {}).get("f1-score"),
            "macro F1": r.get("macro avg", {}).get("f1-score"),
        })
pd.DataFrame(cross_rows).round(4)


In [ ]:
# Probes and auxiliary artifacts - everything else committed under results/
probe_rows = []

fs = json.loads((results / "file_size" / "metrics.json").read_text())
probe_rows.append({"artifact": "file-size probe", "detail": "threshold KB",
                   "value": fs["threshold_kb"], "extra": ""})
for s in fs["sets"]:
    probe_rows.append({"artifact": "file-size rule accuracy", "detail": s["name"],
                       "value": s["accuracy"],
                       "extra": "confusion " + json.dumps(s.get("confusion", {}))})

cp = json.loads((results / "dinov2" / "compression" / "metrics.json").read_text())
probe_rows.append({"artifact": "compression probe", "detail": "accuracy on heldout",
                   "value": cp["accuracy"],
                   "extra": "target " + str(cp.get("target_bytes")) + " bytes"})
probe_rows.append({"artifact": "compression sizes", "detail": "median real before/after KB",
                   "value": round(cp["median_bytes_before"]["real"] / 1024, 1),
                   "extra": round(cp["median_bytes_after"]["real"] / 1024, 1)})
probe_rows.append({"artifact": "compression sizes", "detail": "median fake before/after KB",
                   "value": round(cp["median_bytes_before"]["fake"] / 1024, 1),
                   "extra": round(cp["median_bytes_after"]["fake"] / 1024, 1)})

emb = json.loads((results / "dinov2" / "embedding" / "embedding_classes.json").read_text())
probe_rows.append({"artifact": "embedding", "detail": emb["method"], "value": emb["n_points"],
                   "extra": str(len(emb["classes"])) + " classes"})

pd.DataFrame(probe_rows)
